# Projeto PCD - Treinando pygame com jogo de plataforma

## Importando o que for necessário

In [26]:
import pygame as pg
from pygame import *
import sys
import random

## Cores

In [27]:
white = (255, 255, 255)
red = (255, 0, 0)
green = (0, 255, 0)
blue = (0, 0, 255)
black = (0, 0, 0)

## Código

### Definindo coisas

Nesta parte, são definidas algumas coisas importantes para a funcionalidade do código:

> Constantes:<br>
>> * `WIDGHT`: Define a largura da tela;<br>
>> * `HEIGHT`: Define a altura da tela;<br>
>> * `ACC`: Define a aceleração vertical do player para movimentação mais realista;<br>
>> * `FRIC`: Define um atrito simples para o movimento do player;<br>
>> * `FPS`: Define a taxa de frames por segundo do jogo;<br>

> O que está abaixo de "Criando fontes" serve para definir quais serão as fontes e tamanhos usadas:

>> * `font`: Define uma fonte padrão do tipo "Verdana" de tamanho 60;<br>
>> * `small_font`: Define uma fonte do mesmo tipo de `font`, mas com tamanho 20;<br>

A linha 32 define o nome do jogo, enquanto a linha 20 define o objeto vec, que é um vetor 2d, que será usado em diversas outras ocasiões, como para definir aceleração, veocidade, etc.

In [28]:
####################################################
#----------------------Setup-----------------------#
####################################################

pg.init()

####################################################

##############
# Constantes #
##############
WIDTH = 400
HEIGHT = 450
ACC = 0.5
FRIC = -0.12
FPS = 60

####################################################

vec = pg.math.Vector2

####################################################

##################
# Criando fontes #
##################
font = pg.font.SysFont("Verdana", 60)
small_font = pg.font.SysFont("Verdana", 20)

#########################################################

pg.display.set_caption('The Platform Game')

### Sprites

Esta parte defini tudo sobre as sprites usadas no jogo. Todas elas foram feitas usando classes, uma maneira mais simples de definir um certo objeto. Para tal, todas as classes são "filhas" da classe Sprite do pygame, que tem alguma das funções aqui usadas.

As sprites usadas e suas respecticas funções são:

#### `Player`
Essa classe define o jogador como um todo, dizendo como ele deve se mover, como deve pular, quando não deve, etc. Suas funções são:<br>

##### `__init(self, game)__`

Essa função caracteriza o jogador, definindo sua imagem, sua delimitação, posição - aqui foi usado, inclusive, o objeto vec, para poder facilitar a movimentação e deixar o jogo melhor, definindo sua posição como um vetor no plano da tela - que pode ser alterada facilmente, velocidade, aceleração e vida.

Tudo que é definido nessa função, não necessáriamente será modificado ou usado diretamente nela, pois ela só define suas variáveis principais. Para tal, foram utilizados dois argumentos, `self` e `game`. O argumento `self` é usado para definir variáveis que podem ser usadas em toda a classe, fazendo com que tudo que for criado em tal função, possa ser usado em outras funções - e até mesmo em outras classes. Já o argumento `game` é usado para definir `self.game = game`, que chama a classe `Game`, para que tal classe possa acessar variáveis contidas em `Game`, como, por exemplo, `self.game.platforms`, que chama o conjunto de todas as sprites do tipo plataforma contidas no jogo, enquanto ele roda.

Vars:
* `self.surf`: Irá criar a superfície do jogador, que terá dimensões de 30x30 px. `self.surf.fill()` irá dar cor a essa superfície através de coordenadas RGB. A cor `(0, 255, 255)` dará aspecto ciano ao personagem, que será um quadrado simples.
* `self.rect`: Define a caixa delimitadora do player, usada para verificar colisões, principalmente. Para tal, foi usado `self.surf.get_rect()`, que encontra a caixa mais "justa" para o objeto e o posiciona na tela.
* `self.pos`, um vetor `(x, y)`: Tal variável é um vetor, que define onde o jogador está na tela. Para deixar o jogo mais simples e intuitivo, a superfície - ou a imagem - do jogador, assim como sua caixa delimitadora, serão movidos a partir desse vetor, permitindo que, ao invés de modificar diretamente seu `self.rect`, modifiquemos seu vetor posição, que é mais simples e versátil de ser usado.
* `self.vel`, um vetor `(x, y)`: Este define o vetor de velocidade do player. Geralmente será a partir dele que sua posição será modificada.
* `self.acc`, um vetor `(x, y)`: Esta variável irá definir a aceleração do jogoador.
* `self.jumping`: Isso irá dizer se o jogador está pulando ou não.
* `self.life`: A vida do jogador.

##### `move(self)`

Como o próprio nome diz, tal função define toda a movimentação do jogador. Essa função também usa o argumento self, permitindo que ela acesse todas as variáveis com esse atributo na classe e permitindo que qualquer outra função acesse suas variáveis.

Para tal, primeiramente é difinida uma aceleração inicial de `(0, 0.5)`, ou seja, uma aceleração positiva na direção `y`. Isso é feito para poder simular uma gravidade. Depois, é usado um algorítmo que reconhece uma interação do usuário com o computador, o pressionamento de uma tecla. Com isso, o código define que, se o usuário está apertando a tecla "left" ou "a", a aceleração na direção `x` é `-ACC`, ou seja, para a esquerda. Porém, se o usuário estiver apertando a tecla "right" ou "d", o código define a aceleração na direção `x` como sendo `ACC`, ou seja, para a direita.

Em

```
31  self.acc.x += self.vel.x * FRIC
32  self.vel += self.acc
33  self.pos += self.vel + 0.5 * self.acc
```

é definida a movimentação real do jogador, onde, na linha 31, é adicionado o atrito dinâmico do ambiente, na linha 32 a velocidade é atualizada de acordo com a aceleração - tanto em `x`, como em `y` - e a linha 33 define a posição a partir de uma equação simples.

Já o bloco

```
42  if self.pos.x > WIDTH:
43      self.pos.x = 0
44  if self.pos.x < 0:
45      self.pos.x = WIDTH
```

Cria um efeito de "teletransporte do jogoador caso ele ultrapasse o limite da tela, fazendo com que ele se "teletransporte" para o outro lado, como no jogo Pac-Man.

A linha 47 define o `self.rect` do jogador como a posição `self.pos` dele.|

##### `jump(self)`

Define o pulo do jogador.

Para tal, a função usa de `pg.sprite.spritecollide()` para verificar se o player está encima de uma plataforma e a variável booleana `self.jumping`:

```
50  hits = pg.sprite.spritecollide(
51      self,
52      self.game.platforms,
53      False
54  )
```

`hits` verifica se o player está encima de uma plataforma ou não e em qual está.

```
55  if hits and not self.jumping:
56      self.jumping = True
57
58      self.vel.y = -15
```

Este bloco verifica se o player já está pulando e se ele está encima de algo - porque não faz sentido o player pular enquanto cai e, nesse caso, nem dar um pulo duplo. Caso as o player esteja encima de uma plataforma e não esteja pulando, ele define `self.jumping = True`, para dizer ao código que ele está no meio de um pulo, e define `self.vel.y = -15` fazendo com que o player suba. Note que, por conta do efeito de gravidade que foi colocado, mesmo definindo um valor para a velocidade, o player não sobe a velocidade constante e, após parar de pular, ele cai como que em queda livre.

##### `cancel_jump(self)`

Cancela o pulo - acho que estava meio óbvio... mas é sempre bom dizer.

Essa função define como o pulo irá ser cancelado de maneira muito simples:

```
61  if self.jumping:
62      if self.vel.y < -3:
63          self.vel.y = -3
```

Com isso, quando a função é chamada, caso o jogador esteja em pulo - porque, de novo, não faz sentido cancelar um pulo que não existe - a função define `self.vel.y = -3`, fazendo com que, caso o player ainda esteja subindo, ele tenha uma velocidade de pulo muito baixa, o que faz com que a gravidade faça-o descer.

##### `update`

Essa função básicamente faz com que o player não atravesse as plataformas.

Para funcionar, essa função usa um conceito parecido com a função `jump()`, usando de colisão:

```
67  hits = pg.sprite.spritecollide(
68      self,
69      self.game.platforms,
70      False
71  )
```

Ou seja, aqui ele, de novo, verifica se o jogador está ou não colidindo com uma plataforma, porém essa função também usa as informações da plataforma que está colidindo com o player.

```
73  if self.vel.y > 0 and hits:
74      platform = hits[0]
75
76      if self.pos.y < platform.rect.bottom:
77          self.vel.y = 0
78          self.pos.y = platform.rect.top + 1
79          self.jumping = False
80          self.pos.x += platform.vel.x*platform.direction
```

Primeiro esse bloco verifica se o player está caindo E está colidindo com uma plataforma. Caso isso seja verdadeiro, ele defini a plataforma que está colidindo com `platform = hits[0]`. Então, caso `self.pos.y < platform.rect.bottom`, o código define sua velocidade em `y` como nula, sua posição em y como o topo da plataforma +1 - o +1 serve para que o player não fique colidindo eternamente com a plataforma -, diz para o código que ele parou de pular e, por fim, faz com que o player se mova junto da plataforma, para que ele não caia quando ela se movimentar.

##### `draw(self)`

Essa função só desenha o player na tela.

In [ ]:
##########
# Player #
##########
class Player(pg.sprite.Sprite):
    def __init__(self, game):
        super().__init__()
        self.game = game

        self.surf = pg.Surface((30, 30))
        self.surf.fill((0, 255, 255))
        
        self.rect = self.surf.get_rect()

        self.pos = vec((10, 360))
        self.vel = vec(0,0)
        self.acc = vec(0,0)

        self.jumping = False

        self.life = 100
    
    def move(self):
        self.acc = vec(0, 0.5)
 
        pressed_keys = pg.key.get_pressed()
            
        if pressed_keys[K_LEFT] or pressed_keys[K_a]:
            self.acc.x = -ACC
        if pressed_keys[K_RIGHT] or pressed_keys[K_d]:
            self.acc.x = ACC

        self.acc.x += self.vel.x * FRIC
        self.vel += self.acc
        self.pos += self.vel + 0.5 * self.acc

        if self.pos.x > WIDTH:
            self.pos.x = 0
        if self.pos.x < 0:
            self.pos.x = WIDTH
     
        self.rect.midbottom = self.pos

    def jump(self):
        hits = pg.sprite.spritecollide(
            self,
            self.game.platforms,
            False
        )
        if hits and not self.jumping:
            self.jumping = True

            self.vel.y = -15
    
    def cancel_jump(self):
        if self.jumping:
            if self.vel.y < -3:
                self.vel.y = -3

    # Função de limitação por colisão
    def update(self):
        hits = pg.sprite.spritecollide(
            self,
            self.game.platforms,
            False
        )
        
        if self.vel.y > 0 and hits:
            platform = hits[0]

            if self.pos.y < platform.rect.bottom:
                self.vel.y = 0
                self.pos.y = platform.rect.top + 1
                self.jumping = False
                self.pos.x += platform.vel.x*platform.direction
        
    def draw(self, screen):
        screen.blit(self.surf, self.rect)

In [ ]:
#####################
# Platforms Sprites #
#####################
class Platform(pg.sprite.Sprite):
    def __init__(self, width_max=100, moving=False, speed=0):
        super().__init__()

        self.width_max = width_max
        self.moving = moving
        self.speed = speed
        self.direction = random.randint(-1, 1)
        
        self.surf = pg.Surface(
            (
                random.randint(
                    int(width_max/2),
                    int(width_max)
                ),
                15
            )
        )

        self.surf.fill(green)

        coordenadas = (
            random.randint(0, WIDTH-10),
            random.randint(0, HEIGHT-20)
        )
        self.rect = self.surf.get_rect(
            center = (coordenadas)
        )

        self.vel = vec(speed, 0)
        self.pos = vec(coordenadas)

    def move(self):
        if self.moving:
            self.pos.x += self.vel.x*self.direction

            if self.pos.x <= 0:
                self.direction = 1
            elif self.pos.x >= WIDTH:
                self.direction = -1
            
            self.rect.x = self.pos.x
                
    def draw(self, screen):
        screen.blit(self.surf, self.rect)

#####################

class MinorPlatform(Platform):
    def __init__(self):
        super().__init__(
            width_max=60,
            moving=True,
            speed=3
        )

#####################

class MajorPlatform(Platform):
    def __init__(self):
        super().__init__(
            width_max=80,
            moving=True,
            speed=6
        )

In [29]:
###########
# Enemies #
###########
class Enemy(pg.sprite.Sprite):
    def __init__(self, path, speed=0):
        super().__init__()
        self.image = pg.image.load(path)
        self.rect = self.image.get_rect()
        self.direction = random.randint(-1, 1)

        self.pos = vec((random.randint(0, WIDTH), 0))
        self.vel = vec(speed, 0)

    def move(self):
        self.pos.x += self.vel.x*self.direction

        if self.pos.x <= 0:
            self.direction = 1
        elif self.pos.x >= WIDTH:
            self.direction = -1

        self.rect.x = self.pos.x
        self.rect.y = self.pos.y

    def draw(self, screen):
        screen.blit(self.image, self.rect)

#####################

class BasicEnemy(Enemy):
    def __init__(self):
        super().__init__(
            path="pixel enemy.png",
            speed=3
        )

In [30]:
################################################################
#----------------------------Game------------------------------#
################################################################
class Game:
    def __init__(self):
        self.running = True

        self.screen = pg.display.set_mode((WIDTH, HEIGHT))
        self.clock = pg.time.Clock()
        self.camera_offset = 0
        self.world_height = 0
        self.camera_run = False

        self.all_sprites = pg.sprite.Group()
        self.platforms = pg.sprite.Group()
        self.enemies = pg.sprite.Group()

        self.setup()

    def setup(self):
        self.player = Player(self)
        base_platform = Platform()

        base_platform.surf = pg.Surface((WIDTH, 20))
        base_platform.surf.fill(red)

        base_platform.rect = base_platform.surf.get_rect(
            center=(WIDTH/2, WIDTH-10)
        )
        
        self.platforms.add(base_platform)
        
        self.all_sprites.add(base_platform)
        self.all_sprites.add(self.player)

        for _ in range(5):
            while True:
                pl = Platform()
                # Se a plataforma for correta...
                if not self.check_colision(pl):
                    break

            self.platforms.add(pl)
            self.all_sprites.add(pl)

    def check_colision(self, sprite):
        if pg.sprite.spritecollide(sprite, self.platforms, False):
            return True
        else:
            for entity in self.platforms:
                if entity == sprite:
                    continue
                if (
                    abs(sprite.rect.top - entity.rect.bottom) < 50
                    and
                    (abs(sprite.rect.bottom - entity.rect.top) < 50)
                ):
                    return True
    
        return False

    def plat_gen(self):
        while len(self.platforms) < 6:
            while True:

                if self.world_height <= 1000:
                    PlatformClass = Platform
                elif self.world_height > 1000 and self.world_height <= 3000:
                    PlatformClass = random.choice(
                        [Platform, MinorPlatform]
                    )
                elif self.world_height > 3000 and self.world_height <= 9000:
                    PlatformClass = random.choice(
                        [Platform, MinorPlatform, MajorPlatform]
                    )
                else:
                    chances = [1, 3, 6]
                    PlatformClass = random.choices(
                        [Platform, MinorPlatform, MajorPlatform],
                        weights=chances
                    )[0]

                p = PlatformClass()
                p.rect.center = (
                    random.randrange(0, WIDTH),
                    -(random.randrange(0, 50))
                )

                p.pos = vec(p.rect.center)

                if (
                    not self.check_colision(p)
                    and
                    p.rect.top - self.player.rect.bottom <= 80
                ):
                    break

            self.platforms.add(p)
            self.all_sprites.add(p)

    def enemy_gen(self):
        if self.world_height >= 500 and len(self.enemies) == 0:
            print("entrou na função")

            enemy = BasicEnemy()
            print("criando...")
             
            self.enemies.add(enemy)
            self.all_sprites.add(enemy)
            print("adicionado")

    def input(self):
        for event in pg.event.get():
            if event.type == QUIT:
                self.running = False
            if event.type == pg.KEYDOWN:
                if event.key == pg.K_SPACE:
                    self.player.jump()
            if event.type == pg.KEYUP:
                if event.key == pg.K_SPACE:
                    self.player.cancel_jump()

    def update(self):
        self.player.move()
        self.player.update()

        hits = pg.sprite.spritecollide(self.player, self.enemies, False)
        if hits:
            self.player.life -= 2

        for plat in self.platforms:
            plat.move()

            if plat.rect.top >= HEIGHT:
                plat.kill()

        for enemy in self.enemies:
            enemy.move()
            
            if enemy.rect.top >= HEIGHT:
                enemy.kill()

        # Camera
            # Se o player ultrapassar 2/3 da tela
        if self.player.rect.top <= HEIGHT/3:
            self.camera_run = True
            self.camera_offset = abs(self.player.vel.y)
            self.world_height += self.camera_offset
            self.player.pos.y += self.camera_offset

            for enemy in self.enemies:
                enemy.pos.y += self.camera_offset

            for plat in self.platforms:
                plat.rect.y += self.camera_offset
        
        elif self.camera_run:
            self.camera_offset = 1
            self.world_height += self.camera_offset
            self.player.pos.y += self.camera_offset

            for enemy in self.enemies:
                enemy.pos.y += self.camera_offset

            for plat in self.platforms:
                plat.rect.y += self.camera_offset
        
        # Criando plataformas e inimigo
        self.plat_gen()
        
        self.enemy_gen()

        # Game Over
        if (
            self.player.rect.top >= HEIGHT
            or
            self.player.life <= 0
        ):
            self.running = False

    def draw(self):
        self.screen.fill(black)
        text_height = small_font.render(
            f"Height: {int(self.world_height)}",
            True,
            white
        )
        text_lifes = small_font.render(
            f"Hp: {self.player.life}",
            True,
            green
        )

        self.screen.blit(text_height, (10, 10))
        self.screen.blit(text_lifes, (10, 30))

        for entity in self.all_sprites:
            entity.draw(self.screen)
        
        pg.display.update()

    def run(self):
        while self.running:
            self.clock.tick(FPS)
            self.input()
            self.update()
            self.draw()
        
        pg.quit()
        sys.exit()

game = Game()
game.run()

SystemExit: 